In [ ]:
from dataclasses import dataclass
import math
import random
import numpy as np
import os
from pathlib import Path
from PIL import Image
from collections import Counter
import tqdm
import torch
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster

In [ ]:
DATA_ROOT = '/home/slavik/e202602_eclipse/data'
BRIGHTNESS_MIN = 0.0
BRIGHTNESS_MAX = 1.0
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:

@dataclass
class ImageInfo:
    path: Path
    width: int
    height: int
    avg_brightness: float



In [ ]:
def get_image_infos():
    jpg_files = list(Path(DATA_ROOT).rglob('*.jpg')) + list(Path(DATA_ROOT).rglob('*.JPG'))
    image_infos = []
    for jpg_file in tqdm.tqdm(jpg_files, desc="First scan of images"):
        with Image.open(jpg_file) as img:
            width, height = img.size
            avg_brightness = np.array(img).astype(np.float32).mean() / 255.0
            if BRIGHTNESS_MIN <= avg_brightness <= BRIGHTNESS_MAX:
                image_infos.append(ImageInfo(path=jpg_file, width=width, height=height, avg_brightness=avg_brightness))
    assert len(image_infos) > 0
    for ii in image_infos:
        assert ii.width == image_infos[0].width
        assert ii.height == image_infos[0].height
    image_infos.sort(key=lambda x: x.avg_brightness)
    return image_infos

In [ ]:
N_SECTORS = 360
N_TRIPLETS = 1024
N_CLUSTER = 256
DEBUG_RADIUS_PX = 4
REFINE_ITERATIONS = 3


def refine_moon(img: torch.Tensor, center_i: float, center_j: float) -> tuple[int, int]:
    """
    Refine moon center from image and current center (i, j).
    img: (H, W, 3) float32 [0,1] on GPU.
    Returns (i, j) as tuple of ints.
    """
    assert img.ndim == 3 and img.shape[2] == 3
    H, W = img.shape[0], img.shape[1]
    dev = img.device
    img_size = float(max(H, W))

    gray = img.mean(dim=2)
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=dev).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=dev).view(1, 1, 3, 3)
    g = gray.unsqueeze(0).unsqueeze(0)
    grad_x = torch.nn.functional.conv2d(g, sobel_x, padding=1).squeeze()
    grad_y = torch.nn.functional.conv2d(g, sobel_y, padding=1).squeeze()

    dy = torch.arange(H, device=dev, dtype=torch.float32).view(-1, 1) - center_i
    dx = torch.arange(W, device=dev, dtype=torch.float32).view(1, -1) - center_j
    norm = torch.sqrt(dx * dx + dy * dy).clamp(min=1e-6)
    u_x = dx / norm
    u_y = dy / norm

    angle = torch.atan2(dy, dx)
    sector_id = (torch.floor((angle + math.pi) / (2 * math.pi) * N_SECTORS).long() % N_SECTORS)

    dot_product = grad_x * u_x + grad_y * u_y
    dot_product_flat = dot_product.reshape(-1)
    sector_flat = sector_id.reshape(-1)
    W_t = W

    points_list = []
    for s in range(N_SECTORS):
        mask = sector_flat == s
        if mask.any():
            masked = torch.where(mask, dot_product_flat, torch.tensor(-1e9, device=dev, dtype=torch.float32))
            idx = masked.argmax().item()
            i, j = idx // W_t, idx % W_t
            points_list.append((i, j))

    n_pts = len(points_list)
    if n_pts < 3:
        return (int(round(center_i)), int(round(center_j)))

    def circumcenter(i1, j1, i2, j2, i3, j3):
        x1, y1, x2, y2, x3, y3 = float(j1), float(i1), float(j2), float(i2), float(j3), float(i3)
        D = 2.0 * (x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
        if abs(D) < 1e-10:
            return None
        ox = ((x1 * x1 + y1 * y1) * (y2 - y3) + (x2 * x2 + y2 * y2) * (y3 - y1) + (x3 * x3 + y3 * y3) * (y1 - y2)) / D
        oy = ((x1 * x1 + y1 * y1) * (x3 - x2) + (x2 * x2 + y2 * y2) * (x1 - x3) + (x3 * x3 + y3 * y3) * (x2 - x1)) / D
        oi, oj = oy, ox
        d1 = math.hypot(i1 - oi, j1 - oj)
        d2 = math.hypot(i2 - oi, j2 - oj)
        d3 = math.hypot(i3 - oi, j3 - oj)
        if d1 > img_size or d2 > img_size or d3 > img_size:
            return None
        return (oi, oj)

    circumcenters = []
    while len(circumcenters) < N_TRIPLETS:
        a, b, c = random.sample(range(n_pts), 3)
        i1, j1 = points_list[a]
        i2, j2 = points_list[b]
        i3, j3 = points_list[c]
        cc = circumcenter(i1, j1, i2, j2, i3, j3)
        if cc is not None:
            circumcenters.append(cc)

    pts = np.array(circumcenters, dtype=np.float64)
    Z = linkage(pts, method="complete")
    t_lo, t_hi = 0.0, float(Z[-1, 2])
    for _ in range(60):
        t = (t_lo + t_hi) / 2
        labels = fcluster(Z, t, criterion="distance")
        sizes = np.bincount(labels)
        max_size = int(sizes.max())
        if max_size >= N_CLUSTER:
            t_hi = t
        else:
            t_lo = t
    labels = fcluster(Z, t_hi, criterion="distance")
    sizes = np.bincount(labels)
    which = int(np.argmax(sizes))
    cluster_mask = labels == which
    cluster_pts = pts[cluster_mask]
    ci = float(cluster_pts[:, 0].mean())
    cj = float(cluster_pts[:, 1].mean())
    return (int(round(ci)), int(round(cj)))


def find_moon(img: torch.Tensor) -> tuple[int, int]:
    """
    Find moon center by iteratively refining from image center.
    img: (H, W, 3) float32 [0,1] on GPU. Returns (i, j) as tuple of ints.
    """
    assert img.ndim == 3 and img.shape[2] == 3
    H, W = img.shape[0], img.shape[1]
    center_i, center_j = float(H) / 2, float(W) / 2
    for _ in range(REFINE_ITERATIONS):
        center_i, center_j = refine_moon(img, center_i, center_j)
    i, j = int(center_i), int(center_j)

    img_np = img.cpu().numpy()
    plt.figure(figsize=(12, 8))
    plt.imshow(img_np)
    plt.gca().add_patch(plt.Circle((j, i), DEBUG_RADIUS_PX, color="green", fill=True))
    plt.title(f"Moon center: (i={i}, j={j})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    return (i, j)
    

In [ ]:
image_infos = get_image_infos()
for ii in tqdm.tqdm(image_infos, desc="Finding moon"):
    img = Image.open(ii.path)
    img_arr = torch.from_numpy(np.array(img).astype(np.float32) / 255.0).cuda()
    find_moon(img_arr)
print(len(image_infos))